# 📊 SQL Analysis & Views Showcase (SQLite3 & Python)

Notebook ini digunakan untuk mendemonstrasikan bagaimana kita dapat menguji dan menampilkan analisis relasional SQL secara interaktif menggunakan **SQLite3** dan **Python Pandas**.

### 🛠️ Alur Kerja:
1. Membuat database SQLite lokal (`production.db`).
2. Membaca skema DDL dari `schema.sql` dan membangun struktur tabel relasional.
3. Meng-import data bersih dari folder `data/cleaned/` (CSV) menggunakan Pandas ke dalam tabel database.
4. Meregistrasi fungsi statistik kustom (**STDDEV**) ke SQLite menggunakan Python.
5. Membuat Database Views (`views.sql`) untuk ringkasan laporan.
6. Menjalankan query analitis utama dan menampilkan hasilnya dalam bentuk tabel Pandas DataFrame.

## 1. Setup & Inisialisasi Database

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import math
import os

# Tentukan path database di folder sql
db_path = "production.db"
if os.path.exists(db_path):
    os.remove(db_path)  # Reset DB lama agar bersih

# Sambungkan ke SQLite
conn = sqlite3.connect(db_path)
print("Koneksi ke database SQLite berhasil dibuat.")

Koneksi ke database SQLite berhasil dibuat.


## 2. Registrasi Custom Aggregate Function (STDDEV)
SQLite3 bawaan tidak memiliki fungsi standar deviasi (`STDDEV`). Kita dapat membuatnya dengan mendaftarkan kelas Python baru ke engine SQLite menggunakan `conn.create_aggregate`.

In [2]:
class StdDev:
    def __init__(self):
        self.M = 0.0
        self.S = 0.0
        self.k = 1

    def step(self, value):
        if value is None:
            return
        x = float(value)
        if self.k == 1:
            self.M = x
            self.S = 0.0
        else: 
            old_M = self.M
            self.M += (x - self.M) / self.k
            self.S += (x - old_M) * (x - self.M)
        self.k += 1

    def finalize(self):
        if self.k <= 2:
            return 0.0
        return math.sqrt(self.S / (self.k - 2))  # Sampel Standar Deviasi

conn.create_aggregate("STDDEV", 1, StdDev)
print("Fungsi kustom STDDEV berhasil diregistrasikan ke database SQLite.")

Fungsi kustom STDDEV berhasil diregistrasikan ke database SQLite.


## 3. Eksekusi Skema DDL (`schema.sql`)
Kita membaca script skema relasional dan menjalankannya untuk membuat tabel dimensi dan fakta beserta dengan indeksnya.

In [3]:
# Aktifkan dukungan Foreign Keys di SQLite
conn.execute("PRAGMA foreign_keys = ON;")

with open("schema.sql", "r", encoding="utf-8") as f:
    schema_sql = f.read()

# Penyesuaian sintaks DECIMAL di PostgreSQL ke NUMERIC untuk SQLite
schema_sql = schema_sql.replace("DECIMAL(10,2)", "NUMERIC")
schema_sql = schema_sql.replace("DECIMAL(6,1)", "NUMERIC")

# Eksekusi seluruh script DDL
conn.executescript(schema_sql)
conn.commit()
print("Tabel dimensi dan fakta berhasil dibuat di SQLite.")

Tabel dimensi dan fakta berhasil dibuat di SQLite.


## 4. Import Data CSV ke Database
Kita membaca data hasil pembersihan dari folder `data/cleaned/` lalu menambahkannya ke dalam tabel database yang sesuai.

In [4]:
# Import Tabel Dimensi
pd.read_csv("../data/cleaned/line_dim_cleaned.csv").to_sql("line_dim", conn, if_exists="append", index=False)
pd.read_csv("../data/cleaned/machine_dim_cleaned.csv").to_sql("machine_dim", conn, if_exists="append", index=False)
pd.read_csv("../data/cleaned/shift_dim_cleaned.csv").to_sql("shift_dim", conn, if_exists="append", index=False)
pd.read_csv("../data/cleaned/product_dim_cleaned.csv").to_sql("product_dim", conn, if_exists="append", index=False)

# Import Tabel Fakta
pd.read_csv("../data/cleaned/production_fact_cleaned.csv").to_sql("production_fact", conn, if_exists="append", index=False)

print("Seluruh data CSV berhasil di-import ke database SQLite.")

Seluruh data CSV berhasil di-import ke database SQLite.


### Verifikasi Data Tabel

In [5]:
tables = ["line_dim", "machine_dim", "shift_dim", "product_dim", "production_fact"]
for t in tables:
    row_count = pd.read_sql_query(f"SELECT COUNT(*) AS total_rows FROM {t}", conn)["total_rows"][0]
    print(f"Tabel: {t:<17} | Jumlah Baris: {row_count}")

Tabel: line_dim          | Jumlah Baris: 5
Tabel: machine_dim       | Jumlah Baris: 15
Tabel: shift_dim         | Jumlah Baris: 3
Tabel: product_dim       | Jumlah Baris: 5
Tabel: production_fact   | Jumlah Baris: 24718


## 5. Membuat Database Views (`views.sql`)
Kita membuat views analitik yang dinamis agar siap dikonsumsi langsung oleh tools seperti Power BI.

In [6]:
with open("views.sql", "r", encoding="utf-8") as f:
    views_sql = f.read()

# Penyesuaian sintaks PostgreSQL EXTRACT ke strftime SQLite
views_sql = views_sql.replace(
    "EXTRACT(MONTH FROM production_date)",
    "CAST(strftime('%m', production_date) AS INTEGER)"
)
views_sql = views_sql.replace(
    "EXTRACT(MONTH FROM pf.production_date)",
    "CAST(strftime('%m', pf.production_date) AS INTEGER)"
)

# Eksekusi pembuatan views
conn.executescript(views_sql)
conn.commit()
print("Database Views (v_monthly_executive_kpi, dll) berhasil dibuat di SQLite.")

Database Views (v_monthly_executive_kpi, dll) berhasil dibuat di SQLite.


## 6. Uji Coba Query Analisis (Showcase)
Mari jalankan beberapa query analisis utama untuk memverifikasi fungsionalitas dan membuktikan kesimpulan data.

### Query 1: Total Output & Efisiensi Keseluruhan

In [7]:
q1 = """
SELECT
    COUNT(*) AS total_records,
    SUM(output_qty) AS total_output,
    SUM(target_qty) AS total_target,
    SUM(defect_qty) AS total_defect,
    ROUND(SUM(output_qty) * 100.0 / SUM(target_qty), 1) AS achievement_pct,
    ROUND(SUM(defect_qty) * 100.0 / SUM(output_qty), 2) AS defect_rate_pct,
    ROUND(AVG(downtime_minutes), 1) AS avg_downtime
FROM production_fact;
"""
pd.read_sql_query(q1, conn)

,total_records,total_output,total_target,total_defect,achievement_pct,defect_rate_pct,avg_downtime
0,24718,961992,1052628,31956,91.4,3.32,21.4


### Query 2: Defect Rate per Shift Kerja (Mendeteksi Shift Kritis)

In [8]:
q3 = """
SELECT
    s.shift_name,
    s.start_time,
    s.end_time,
    SUM(pf.output_qty) AS total_output,
    SUM(pf.defect_qty) AS total_defect,
    ROUND(SUM(pf.defect_qty) * 100.0 / SUM(pf.output_qty), 2) AS defect_rate_pct,
    CASE
        WHEN SUM(pf.defect_qty) * 100.0 / SUM(pf.output_qty) > 4 THEN 'HIGH'
        WHEN SUM(pf.defect_qty) * 100.0 / SUM(pf.output_qty) > 3 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS defect_category
FROM production_fact pf
JOIN shift_dim s ON pf.shift_id = s.shift_id
GROUP BY s.shift_name, s.start_time, s.end_time
ORDER BY defect_rate_pct DESC;
"""
pd.read_sql_query(q3, conn)

,shift_name,start_time,end_time,total_output,total_defect,defect_rate_pct,defect_category
0,Shift Malam,22:00,06:00,292525,14869,5.08,HIGH
1,Shift Siang,14:00,22:00,326243,9517,2.92,LOW
2,Shift Pagi,06:00,14:00,343224,7570,2.21,LOW


### Query 3: Analisis Month-on-Month (MoM) Growth Output Produksi

In [9]:
q10 = """
SELECT
    month_num,
    monthly_output,
    LAG(monthly_output) OVER (ORDER BY month_num) AS prev_month_output,
    monthly_output - LAG(monthly_output) OVER (ORDER BY month_num) AS mom_change,
    ROUND(
        (monthly_output - LAG(monthly_output) OVER (ORDER BY month_num)) * 100.0
        / LAG(monthly_output) OVER (ORDER BY month_num), 1
    ) AS mom_change_pct
FROM (
    SELECT
        CAST(strftime('%m', production_date) AS INTEGER) AS month_num,
        SUM(output_qty) AS monthly_output
    FROM production_fact
    GROUP BY CAST(strftime('%m', production_date) AS INTEGER)
) monthly_data
ORDER BY month_num;
"""
pd.read_sql_query(q10, conn)

,month_num,monthly_output,prev_month_output,mom_change,mom_change_pct
0,7,176714,NaN,NaN,NaN
1,8,170244,176714.0,-6470.0,-3.7
2,9,160505,170244.0,-9739.0,-5.7
3,10,167325,160505.0,6820.0,4.2
4,11,149804,167325.0,-17521.0,-10.5
5,12,137400,149804.0,-12404.0,-8.3


### Query 4: Anomali Output Harian (Deteksi Jatuh > 1.5 StdDev)

In [10]:
q13 = """
WITH daily_output AS (
    SELECT
        production_date,
        SUM(output_qty) AS total_daily_output
    FROM production_fact
    GROUP BY production_date
),
stats AS (
    SELECT
        AVG(total_daily_output) AS avg_daily,
        AVG(total_daily_output) - 1.5 * STDDEV(total_daily_output) AS lower_bound
    FROM daily_output
    -- Filter Hari Kerja (Senin s.d Jumat) menggunakan strftime '%w' (1 s.d 5)
    WHERE CAST(strftime('%w', production_date) AS INTEGER) BETWEEN 1 AND 5
)
SELECT
    d.production_date,
    d.total_daily_output,
    ROUND(s.avg_daily, 0) AS avg_daily,
    ROUND(d.total_daily_output - s.avg_daily, 0) AS deviation
FROM daily_output d
CROSS JOIN stats s
WHERE d.total_daily_output < s.lower_bound
ORDER BY d.total_daily_output ASC;
"""
pd.read_sql_query(q13, conn)

,production_date,total_daily_output,avg_daily,deviation
0,2024-12-22,1635,6213.0,-4578.0
1,2024-12-29,1679,6213.0,-4534.0
2,2024-12-01,1719,6213.0,-4494.0
3,2024-12-15,1720,6213.0,-4493.0
4,2024-12-08,1736,6213.0,-4477.0
5,2024-11-10,1821,6213.0,-4392.0
6,2024-11-17,1824,6213.0,-4389.0
7,2024-11-03,1848,6213.0,-4365.0
8,2024-11-24,1899,6213.0,-4314.0
9,2024-10-20,1911,6213.0,-4302.0


## 7. Membaca Hasil Database Views

### View: Prioritas Pemeliharaan Mesin

In [11]:
pd.read_sql_query("SELECT * FROM v_machine_maintenance_priority WHERE maintenance_recommendation LIKE 'PRIORITAS%'", conn)

,line_name,machine_name,machine_type,machine_age,avg_downtime_min,avg_output_units,batch_count,maintenance_recommendation
0,Line A,SMT-01,SMT Placement,6,35.5,40.0,1648,PRIORITAS TINGGI - Perlu preventive maintenanc...
1,Line B,WAVE-01,Wave Soldering,7,37.0,38.0,1646,PRIORITAS TINGGI - Perlu preventive maintenanc...
2,Line B,WAVE-02,Wave Soldering,4,22.5,39.0,1642,PRIORITAS SEDANG - Jadwalkan maintenance rutin
3,Line C,AOI-01,Inspection,5,24.1,41.0,1647,PRIORITAS SEDANG - Jadwalkan maintenance rutin
4,Line C,ICT-01,Testing,6,34.7,40.0,1645,PRIORITAS TINGGI - Perlu preventive maintenanc...
5,Line D,REFLOW-01,Reflow Oven,8,34.5,36.0,1653,PRIORITAS TINGGI - Perlu preventive maintenanc...
6,Line D,REFLOW-02,Reflow Oven,4,23.0,38.0,1646,PRIORITAS SEDANG - Jadwalkan maintenance rutin
7,Line E,PRESS-01,Press,5,22.9,36.0,1649,PRIORITAS SEDANG - Jadwalkan maintenance rutin


### View: Analisis Defect & Kerugian Keuangan per Produk

In [12]:
pd.read_sql_query("SELECT * FROM v_product_defect_summary ORDER BY total_defect_cost_idr DESC", conn)

,product_name,category,total_output,total_defect,defect_rate_pct,cost_per_unit_idr,total_defect_cost_idr
0,Control Board,Controller,194986,12858,6.59,120000,1.542960e+09
1,Power Supply Unit,Power,193243,3837,1.99,78000,2.992860e+08
2,Sensor Module,Component,193856,6171,3.18,45000,2.776950e+08
3,LED Panel,Display,185397,4162,2.24,35000,1.456700e+08
4,PCB Assembly,Assembly,194510,4928,2.53,25000,1.232000e+08


In [13]:
# Tutup Koneksi Database
conn.close()
print("Koneksi database berhasil ditutup secara aman.")

Koneksi database berhasil ditutup secara aman.
